In [1]:
from ultralytics import YOLO
import cv2
import cvzone
import math
from sort import *

In [2]:
model = YOLO('../model/yolov8n.pt')

In [16]:
cap = cv2.VideoCapture("car.mp4")
mask = cv2.imread('car_edited.png')
KalmanBoxTracker.count = 0
tracker = Sort(max_age=20)
totalcount = []
limit = [166,201,584,201]
pasue= False
while True:
    ret, frame = cap.read()


    if not ret:
        break

    frame_after_edit = cv2.bitwise_and(frame,mask)
    results = model(frame_after_edit,stream=True)
    detections = np.empty((0,5))

    for r in results:
        for box in r.boxes:

            x1,y1,x2,y2 = box.xyxy[0]
            x1,y1,x2,y2 = int(x1),int(y1),int(x2),int(y2)

            w,h = x2-x1,y2-y1


            conf  = math.ceil((box.conf[0] * 100))/100
            cls = int(box.cls[0])
            currentclass = model.names[cls]
            if (currentclass == 'car' or currentclass == 'truck') and conf > 0.4:
                #cvzone.cornerRect(frame,(x1,y1,w,h),l=9)
                #cvzone.putTextRect(frame,f'{currentclass} {conf}',(max(0,x1),max(35,y1)),scale=0.7,thickness=1)
                currentArray = np.array([x1,y1,x2,y2,conf])
                detections = np.vstack((detections,currentArray))


            
    resultTracker = tracker.update(detections)
    cv2.line(frame,(limit[0],limit[1]),(limit[2],limit[3]),color=(0,0,255),thickness=5)
    for result in resultTracker:
        x1,y1,x2,y2,id = result
        x1,y1,x2,y2 = int(x1),int(y1),int(x2),int(y2)
        print(result)
        w,h = x2-x1,y2-y1


        #print(result)
        cvzone.cornerRect(frame,(x1,y1,w,h),l=9,rt=2,colorR=(255,0,0))
        cvzone.putTextRect(frame,f'{int(id)}',(max(0,x1),max(35,y1)),scale=0.7,thickness=1)

        cx,cy = x1+w//2,y1+h//2
        cv2.circle(frame,(cx,cy),5,(255,0,255),-1)
        if limit[0] < cx < limit[2] and limit[1]-5 < cy < limit[1]+5:
            if totalcount.count(id) == 0:
                totalcount.append(id)
                cv2.line(frame,(limit[0],limit[1]),(limit[2],limit[3]),color=(0,255,0),thickness=5)


    
    
    cvzone.putTextRect(frame,f'Counter: {len(totalcount)}',(50,50))

    cv2.imshow("Detection", frame)
    cv2.imshow('Mask',frame_after_edit)
    cv2.waitKey(1)
    if cv2.waitKey(2) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

0: 384x640 11 cars, 1 truck, 47.2ms
Speed: 7.2ms preprocess, 47.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
[        379         286         554         358           9]
[        368         176         445         204           8]
[        392         144         425         175           7]
[        208         168         291         237           6]
[         83         326         197         359           5]
[        191         217         273         300           4]
[        365         192         465         290           3]
[        339         145         404         204           2]
[        126         228         237         329           1]

0: 384x640 12 cars, 1 truck, 13.2ms
Speed: 0.9ms preprocess, 13.2ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)
[     379.17      286.93      552.83      358.07           9]
[     367.67      176.12      445.33      204.88           8]
[     391.12      143.89      424.88      175.11   